In [1]:
import os

In [2]:
%pwd

'd:\\PredictBot-Score-MLOps\\research'

In [3]:
os.chdir('..')

In [4]:
%pwd

'd:\\PredictBot-Score-MLOps'

In [5]:
from dataclasses import dataclass
from pathlib import Path
from src.predictor_bot_score.config.configuration import yaml_load , create_directories
from src.predictor_bot_score.logger import logger
from src.predictor_bot_score.constants import CONFIG_PATH
from src.predictor_bot_score.utils.model_factory import get_model , get_fit_kwargs , get_mlflow_logger , MODEL_REGISTRY
from sklearn.metrics import mean_absolute_error
import os
import numpy as np
import pandas as pd
import glob
import mlflow
import lightgbm
import pickle
from datetime import datetime
import sqlite3

In [6]:
@dataclass(frozen=True)
class ModelTrainingConfig:
    train_data_path       : Path
    val_data_path         : Path
    model_dir             : Path
    active_model_strategy : str
    models                : dict
    features              : list[str]
    target_column         : str
    baseline_mae          : float
    promotion_criteria    : dict
    mlflow_experiment     : str
    mlflow_tracking_uri   : str

In [7]:
class config_manager:

    def __init__(self, config = CONFIG_PATH):

        self.config = yaml_load(config)
        
        create_directories([self.config.artifacts_root])

    def get_model_training_config(self) -> ModelTrainingConfig:

        config = self.config.model_training

        create_directories([config.model_dir])

        return ModelTrainingConfig(
        train_data_path       = Path(config.train_data_path),
        val_data_path         = Path(config.val_data_path),
        model_dir             = Path(config.model_dir),
        active_model_strategy = config.active_model_strategy,
        models                = dict(config.models),
        features              = list(config.features),
        target_column         = config.target_column,
        baseline_mae          = float(config.baseline_mae),
        promotion_criteria    = dict(config.promotion_criteria),
        mlflow_experiment     = config.mlflow.experiment_name,
        mlflow_tracking_uri   = config.mlflow.tracking_uri,
    )
        

In [ ]:
class Model_Building :

    def __init__(self, config : ModelTrainingConfig):
        self.config = config
        self.train , self.val = self._read_data()

    def _get_files(self , folder:Path ,prefix :str):
        files = glob.glob(os.path.join(folder ,f"{prefix}_*.csv"))
        if not files:
            raise FileNotFoundError(f"No {prefix} file found in {folder}")
        return max(files , key=os.path.getmtime)
    

    def _read_data(self):
        try:
            logger.info("=" * 50)
            logger.info("Reading train /  test splits")
            logger.info("=" * 50)

            train_file = self._get_files(self.config.train_data_path, "train")
            val_file   = self._get_files(self.config.val_data_path,   "val")

            train = pd.read_csv(train_file)
            val   = pd.read_csv(val_file)
            
            logger.info(f"Train : {len(train)} rows")
            logger.info(f"Val   : {len(val)} rows")

            return train, val

        except FileNotFoundError as e:
            logger.error(f"Split file not found: {e}")
            raise

        except Exception as e:
            logger.error(f"Failed to read splits: {str(e)}")
            raise

    def prepare_data(self ,df :pd.DataFrame):
        try:
            X = df[self.config.features]
            y = df[self.config.target_column]

            return X ,y
        except Exception as e:
            raise e 
        
    def _smape(self, actual , predicted ):
        try:
            sampe_result = float(
                100 * np.mean(
                    2 * np.abs(predicted - actual) /
                    (np.abs(actual) + np.abs(predicted) + 1e-8)
                )
            )

            return sampe_result
        except Exception as e:
            logger.error(f"SMAPE calculation failed: {str(e)}")
            raise 
    
    def model_training(self):
        try:
            X_train, y_train = self.prepare_data(self.train)
            X_val,   y_val   = self.prepare_data(self.val)

            trained_models = {}

            for model_name, cfg in self.config.models.items():
                model_type   = cfg["type"]
                model_params = cfg["params"]

                if model_type not in MODEL_REGISTRY:
                    logger.warning(
                        f"Skipping {model_name}: '{model_type}' not in registry"
                    )
                    continue

                logger.info("")
                logger.info(f"--- Training {model_name} ({model_type}) ---")
                logger.info("-" * 50)

                model      = get_model(model_type, model_params)
                eval_set   = [(X_val, y_val)]
                fit_kwargs = get_fit_kwargs(model_type, eval_set)

                model.fit(X_train, y_train, **fit_kwargs)

                val_pred  = model.predict(X_val)
                val_mae   = float(mean_absolute_error(y_val, val_pred))
                val_smape = self._smape(y_val.values, val_pred)

                trained_models[model_name] = {
                    "model"      : model,
                    "model_type" : model_type,
                    "params"     : model_params,
                    "val_mae"    : val_mae,
                    "val_smape"  : val_smape,
                }

                logger.info(f"{model_name} VAL MAE   : {val_mae:.6f}")
                logger.info(f"{model_name} VAL SMAPE : {val_smape:.2f}%")
                logger.info(f"PASSED - {model_name} trained")

            return trained_models

        except Exception as e:
            logger.error(f"Training failed: {str(e)}")
            raise
    
    def save_model(self,model , model_name):

        try:
            path = self.config.model_dir
            time_stamp = datetime.now().strftime("%Y_%m_%d_%H_%M")
            model_path = os.path.join(path,f"{model_name}__{time_stamp}.pkl")

            os.makedirs(path , exist_ok=True)
            
            with open(model_path ,"wb") as f:
                pickle.dump(model , f)
                f.close()
        
            logger.info(f"Model saved  {model_path}")
            logger.info("PASSED - Model saved")
            logger.info("-" * 50)

            return model_path

        except Exception as e:
            logger.error(f"Failed to save model: {str(e)}")
            raise

    # ── log to mlflow ─────────────────────────────────────
    def log_to_mlflow(self, model, model_path):
        try:
            logger.info("")
            logger.info("fLOGGING {model_name} TO MLFLOW")
            logger.info("-" * 50)

            mlflow.set_tracking_uri(self.config.mlflow_tracking_uri)
            mlflow.set_experiment(self.config.mlflow_experiment)

            with mlflow.start_run():

                mlflow.log_params(self.config.lgbm_params)

                mlflow.set_tag("stage",      "Staging")
                mlflow.set_tag("trained_at", datetime.now().isoformat())
                mlflow.set_tag("model_path", model_path)


                mlflow.lightgbm.log_model(
                    model,
                    "model",
                    registered_model_name="Predicted_model_name"

                )
            logger.info("PASSED - MLflow logging complete")
            logger.info("-" * 50)

        except Exception as e:
            logger.error(f"MLflow logging failed: {str(e)}")
            raise

    def run(self):
        
        try:
            logger.info("=" * 50)
            logger.info("MODEL TRAINING PIPELINE STARTED")
            logger.info("=" * 50)

            trained_models = self.model_training()

            for model_name , model in trained_models.items():

                self.save_model(model , model_name=model_name)
                
            

            logger.info("=" * 50)
            logger.info("MODEL TRAINING PIPELINE COMPLETE")
            logger.info("=" * 50)

        except Exception as e:
            logger.error(f"Model training pipeline failed: {str(e)}")
            raise



             

   

In [20]:
xc = config_manager()
xc = xc.get_model_training_config()
xc = Model_Building(xc)
xc.run()


[2026-06-18 10:56:43,978: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-18 10:56:43,982: INFO: common: Directory created (or already exists) at: artifacts]
[2026-06-18 10:56:43,984: INFO: common: Directory created (or already exists) at: artifacts/models_directory/models]
[2026-06-18 10:56:43,986: INFO: 47318016: ==================================================]
[2026-06-18 10:56:43,988: INFO: 47318016: Reading train /  test splits]
[2026-06-18 10:56:43,990: INFO: 47318016: ==================================================]


[2026-06-18 10:56:44,048: INFO: 47318016: Train : 28426 rows]
[2026-06-18 10:56:44,049: INFO: 47318016: Val   : 4061 rows]
[2026-06-18 10:56:44,051: INFO: 47318016: ==================================================]
[2026-06-18 10:56:44,052: INFO: 47318016: MODEL TRAINING PIPELINE STARTED]
[2026-06-18 10:56:44,053: INFO: 47318016: ==================================================]
[2026-06-18 10:56:44,106: INFO: 47318016: ]
[2026-06-18 10:56:44,107: INFO: 47318016: --- Training lightgbm_v1 (lightgbm) ---]
[2026-06-18 10:56:44,108: INFO: 47318016: --------------------------------------------------]
[200]	valid_0's l1: 0.019045
[400]	valid_0's l1: 0.0172147
[600]	valid_0's l1: 0.0169749
[800]	valid_0's l1: 0.0168863
[1000]	valid_0's l1: 0.0167948
[1200]	valid_0's l1: 0.0167169
[1400]	valid_0's l1: 0.0166661
[1600]	valid_0's l1: 0.0165971
[1800]	valid_0's l1: 0.0165488
[2000]	valid_0's l1: 0.0165087
[2026-06-18 10:56:47,699: INFO: 47318016: lightgbm_v1 VAL MAE   : 0.016509]
[2026-06-18 